# Models

In [ ]:
# @title Installs (Minimal für Google GenAI, OpenAI SDK, LiteLLM & Model Garden)
############################################

# 1. Benötigte Pakete für Models installieren (ohne ADK)
%pip install --upgrade --quiet google-genai google-cloud-aiplatform openai litellm

# 2. Versionen kurz prüfen
%pip show google-genai google-cloud-aiplatform openai litellm


In [2]:
# @title Imports (Minimal für Models)
############################################

import os
import warnings
warnings.filterwarnings('ignore')

# 1. Google Cloud Auth & Vertex AI
import google.auth
import google.auth.transport.requests
import vertexai
from vertexai import model_garden

# 2. Google GenAI SDK
from google import genai
from google.genai import types

# 3. OpenAI SDK (für Vertex AI OpenAI-kompatiblen Endpunkt)
from openai import OpenAI

# 4. LiteLLM
import litellm

print("✅ Alle Imports erfolgreich geladen!")


✅ Alle Imports erfolgreich geladen!


In [3]:
# @title Environmental Variables & GCP Setup
############################################

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"

# These tell the underlying google-genai client and SDK to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Required by LiteLLM for Vertex AI partner models
os.environ["VERTEXAI_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_LOCATION"] = LOCATION

# Initialize Vertex AI SDK & Client via Application Default Credentials (ADC)
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

print("✅ Erfolgreich über Application Default Credentials (ADC) verbunden.")

✅ Erfolgreich über Application Default Credentials (ADC) verbunden.


In [ ]:
# @title Google SDK (Fast Track)

from google import genai

client = genai.Client(vertexai=True, project="deltorobarba", location="global")

response = client.models.generate_content(
    model="gemma-4-26b-a4b-it-maas", # gemini-4-argon, gemini-3.8-flash
    contents="Hi, welche Fähigkeiten hast du? Bitte antworte kurz.",
)
print(response.text)

Ich bin Gemma 4, ein Large Language Model von Google DeepMind. Meine Fähigkeiten umfassen:

* **Textverarbeitung:** Ich kann Texte verstehen, zusammenfassen, übersetzen, schreiben und Fragen beantworten.
* **Bildverständnis:** Ich kann Bilder analysieren und beschreiben.
* **Audioverarbeitung:** In den 2B- und 4B-Versionen meiner Modellfamilie kann ich auch Audioeingaben verarbeiten.
* **Wissensstand:** Mein Wissen reicht bis Januar 2025.

Ich gebe ausschließlich Text aus und kann keine Bilder generieren.


In [ ]:
# @title Google SDK (Full Track)

# https://github.com/se02035/local-llm-proxy it helps to spin up a litellm proxy with local ollama. also supports a public endpoint using ngrok

# Locations: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/locations
# Google Models: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models
# Versions: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions

# client = genai.Client(http_options=HttpOptions(api_version="v1")) --> only for Google models directly
# https://github.com/GoogleCloudPlatform/generative-ai/blob/main/sdk/intro_genai_sdk.ipynb
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/open-models/use-maas

from google import genai
from google.genai.types import GenerateContentConfig
from google.genai import types

PROJECT_ID="deltorobarba"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# Closed Models
geminipro      = "google/gemini-4-argon"
geminiflash    = "google/gemini-3.8-flash"
grok            = "xai/grok-4.1-fast-reasoning"
claude          = "anthropic/claude-opus-4-7"

# Open Models (kimi, qwen, minimax not supported anymore)
gemma           = "google/gemma-4-26b-a4b-it-maas"
deepseek        = "deepseek-ai/deepseek-v3.2-maas"
glm             = "zai-org/glm-5-maas"
llama           = "meta/llama-3.3-70b-instruct-maas"   # LOCATION = "us-central1"

system_instruction = """You are a helpful machine learning advisor and answer briefly."""
prompt = """Hi, which LLM are you? What is your name and what are your capabilities? Please answer briefly."""

generate_content_config = types.GenerateContentConfig(
    temperature=0.4,
    top_p=0.95,
    top_k=20,
    candidate_count=1,
    seed=5,
    max_output_tokens=120,
    stop_sequences=["STOP!"],
    presence_penalty=0.0,
    frequency_penalty=0.0,
    system_instruction=system_instruction,)

response = client.models.generate_content(
    model=geminipro,                       # <--- ⚠️ Define model here
    contents=prompt,
    config=generate_content_config,
)
print("✅ Success")
print(response.text)

✅ Success
Hello! I am **Gemini 4 Argon**, a large language model built by Google. 

As your machine learning advisor, my capabilities include:
* **Code Generation & Debugging:** Writing and optimizing ML code in Python, PyTorch, TensorFlow, and scikit-learn.
* **Architecture & Algorithm Selection:** Recommending the right models (from classical ML to deep learning) for your specific data and goals.
* **Data & MLOps Advice:** Assisting with data preprocessing, feature engineering, hyperparameter tuning, and model deployment strategies.
* **Concept Explanation


In [ ]:
# @title OpenAI SDK for LLMs on Google Cloud
#################################################

from openai import OpenAI

PROJECT  = "deltorobarba"
LOCATION = "global"

# ⚠️ Accept User Agreements per Model!
grok = "xai/grok-4.1-fast-reasoning"        # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/partner-models/grok/grok-4-1-fast
gemma = "google/gemma-4-26b-a4b-it-maas"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/google/gemma-4-26b-a4b-it
gemini = "google/gemini-3.1-pro-preview"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/3-1-pro
deepseek = "deepseek-ai/deepseek-v4.1-flash-maas" # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32

# Get access token from Application Default Credentials (run `gcloud auth application-default login` once beforehand)
credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())

# Point OpenAI SDK at Vertex's OpenAI-compatible endpoint
client = OpenAI(
    base_url=f"https://aiplatform.googleapis.com/v1/projects/{PROJECT}/locations/{LOCATION}/endpoints/openapi",
    api_key=credentials.token,
)

# Call selected model like any OpenAI model
response = client.chat.completions.create(
    model=deepseek,
    messages=[{"role": "user", "content": "Hi, which LLM are you and which capabilities do you have? Answer me in Chinese."}],
    temperature=0.2,
)

print("✅ Success")
print(response.choices[0].message.content)

✅ Success
你好！我是 **DeepSeek**，由深度求索公司创造的AI助手。😊

以下是我的一些主要能力和特点：

**基本信息：**
- 知识截止日期：2026年2月
- 支持中文和多种语言
- 完全免费使用

**核心能力：**
1. **文本对话**：可以回答问题、写作、翻译、编程、分析等
2. **文件处理**：支持上传图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息
3. **长文本处理**：上下文长度达1M，可以一次性处理像《三体》三部曲这样体量的书籍
4. **联网搜索**：支持联网搜索功能（需要在Web/App端手动开启）
5. **语音输入**：App端支持语音输入
6. **阅读链接**：可以阅读网页链接内容

**注意事项：**
- 我支持图片输入，可以识别和分析图片中的可见信息，比如物体、场景、文字等
- 联网搜索需要用户手动开启

有什么我可以帮你的吗？无论是学习、工作还是生活问题，都可以问我！🌟


ADK nutzt unter der Haube automatisch google-genai und deine Umgebungsvariablen (GOOGLE_GENAI_USE_VERTEXAI="1", GOOGLE_CLOUD_PROJECT="deltorobarba", GOOGLE_CLOUD_LOCATION="global"). Native Features wie Google Search Grounding, Vertex Search RAG, URL Context und Memory Bank funktionieren so ohne jede Zusatzkonfiguration.

* Hinweis für Google ADK (agents.ipynb):
* Falls ein Modell in einem ADK-Agenten genutzt werden soll, dort nutzen:
* from google.adk.models.lite_llm import LiteLlm
* deepseek_agent_model = LiteLlm(model=deepseek, vertex_project="deltorobarba", vertex_location="global")


**Warum das Google SDK (google-genai) dem OpenAI SDK weit überlegen ist:**

* Kein OAuth2-Token-Management: Beim OpenAI SDK musst du manuell per google.auth.default() ein Google-Token anfordern und per credentials.refresh() erneuern (Token verfällt nach 60 Minuten!). Das Google SDK übernimmt die Authentifizierung über deine lokalen Application Default Credentials (ADC) vollautomatisch im Hintergrund.
* Keine URL-Bastelei: Kein manuelles Zusammensetzen von https://aiplatform.googleapis.com/v1/projects/.../endpoints/openapi.
* Voller Feature-Umfang: Über den OpenAI-Compatibility-Layer verlierst du native Gemini-Funktionen (wie multimodales Audio/Video/PDF-Handling, Google Search Grounding, Thinking-Budgets, Structured Outputs via Pydantic und Code Execution).
* Funktioniert auch für Model Garden (MaaS): Sogar Open Models aus Model Garden (wie Gemma 4, Llama 3.3, DeepSeek etc.) lassen sich über vertexai=True direkt mit demselben genai.Client ansprechen (genau wie du es in Zelle 4 von models.ipynb bereits aufgebaut hast).

In [24]:
# @title 📊 Model Benchmark: Speed (TTFT, Total Time, Tokens/sec) & LLM-as-a-Judge
################################################################################

import time
import pandas as pd
from google import genai

# 1. Client initialisieren
PROJECT_ID = "deltorobarba"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# 2. Zu vergleichende Modelle & Test-Prompts definieren
models_to_test = [
    "gemini-3.8-flash",
    "deepseek-ai/deepseek-v4.1-flash-maas",  # oder Partner-MaaS Modelle wie deepseek-v4.1-flash-maas
]

test_prompts = {
    "Reasoning & Math": "Ein Zug fährt mit 120 km/h von A nach B (300 km). Ein zweiter Zug startet 30 Minuten später mit 150 km/h von B nach A. Wann und wo begegnen sie sich? Erkläre Schritt für Schritt.",
    "Code Generation": "Schreibe eine Python-Funktion mit Type Hints und Docstring, die einen Text einliest und die 3 häufigsten Wörter (ohne Stopwords) zurückgibt.",
    "Summarization": "Fasse die Kernunterschiede zwischen RAG (Retrieval-Augmented Generation) und Fine-Tuning in genau 3 prägnanten Bullet Points zusammen."
}

# 3. Benchmark-Funktion mit Streaming (misst TTFT + Generation Time)
def run_benchmark(models, prompts):
    results = []
    outputs = {}

    for prompt_name, prompt_text in prompts.items():
        print(f"\n🚀 Teste Kategorie: {prompt_name}...")
        outputs[prompt_name] = {}

        for model in models:
            model_short = model.split("/")[-1]
            try:
                start_time = time.perf_counter()
                ttft = None
                chunks = []
                last_chunk = None

                # Streaming für Time-To-First-Token (TTFT)
                stream = client.models.generate_content_stream(model=model, contents=prompt_text)
                for chunk in stream:
                    if ttft is None:
                        ttft = time.perf_counter() - start_time
                    if chunk.text:
                        chunks.append(chunk.text)
                    last_chunk = chunk

                total_time = time.perf_counter() - start_time
                full_text = "".join(chunks)
                outputs[prompt_name][model] = full_text

                # Token Usage aus Metadaten ermitteln
                usage = getattr(last_chunk, "usage_metadata", None)
                in_tokens = usage.prompt_token_count if usage else None
                out_tokens = usage.candidates_token_count if usage else None

                # Durchsatz (Tokens/s nach erstem Token)
                gen_time = total_time - (ttft or 0)
                tps = round(out_tokens / gen_time, 1) if (out_tokens and gen_time > 0) else "N/A"

                results.append({
                    "Kategorie": prompt_name,
                    "Modell": model_short,
                    "TTFT (s)": round(ttft, 3) if ttft else None,
                    "Gesamtzeit (s)": round(total_time, 3),
                    "Output Tokens": out_tokens,
                    "Tokens/Sek (TPS)": tps,
                })
                print(f"   ✅ {model_short}: TTFT={round(ttft, 2)}s, Gesamt={round(total_time, 2)}s, TPS={tps}")
            except Exception as e:
                print(f"   ❌ Fehler bei {model}: {e}")

    df = pd.DataFrame(results)
    return df, outputs

# Ausführen
df_results, collected_outputs = run_benchmark(models_to_test, test_prompts)
display(df_results)

# 4. Optional: LLM-as-a-Judge (Ein Richter-Modell bewertet beide Antworten blind)
def judge_pairwise(prompt, text_a, text_b, judge_model="gemini-3.8-flash"):
    judge_prompt = f"""
Du bist ein unparteiischer KI-Schiedsrichter. Bewerte die folgenden zwei Antworten auf die gegebene Frage.
Frage: {prompt}

--- Antwort A ---
{text_a}

--- Antwort B ---
{text_b}

Bewerte beide Antworten nach Genauigkeit, Struktur und Logik (Punkte 1-10).
Erkläre kurz deine Entscheidung und küre einen Gewinner (A, B oder Unentschieden).
"""
    response = client.models.generate_content(model=judge_model, contents=judge_prompt)
    print("\n⚖️ Richter-Urteil:")
    print(response.text)

# Beispiel: Richter für "Reasoning & Math" aufrufen
# judge_pairwise(test_prompts["Reasoning & Math"], collected_outputs["Reasoning & Math"][models_to_test[0]], collected_outputs["Reasoning & Math"][models_to_test[1]])



🚀 Teste Kategorie: Reasoning & Math...
   ✅ gemini-3.8-flash: TTFT=145.03s, Gesamt=153.02s, TPS=129.6
   ✅ deepseek-v4.1-flash-maas: TTFT=1.45s, Gesamt=15.21s, TPS=51.7

🚀 Teste Kategorie: Code Generation...
   ✅ gemini-3.8-flash: TTFT=212.72s, Gesamt=222.34s, TPS=157.1
   ✅ deepseek-v4.1-flash-maas: TTFT=0.88s, Gesamt=13.53s, TPS=91.1

🚀 Teste Kategorie: Summarization...
   ✅ gemini-3.8-flash: TTFT=237.1s, Gesamt=238.95s, TPS=102.1
   ✅ deepseek-v4.1-flash-maas: TTFT=2.34s, Gesamt=12.82s, TPS=12.0


,Kategorie,Modell,TTFT (s),Gesamtzeit (s),Output Tokens,Tokens/Sek (TPS)
0,Reasoning & Math,gemini-3.8-flash,145.030,153.022,1036,129.6
1,Reasoning & Math,deepseek-v4.1-flash-maas,1.450,15.211,712,51.7
2,Code Generation,gemini-3.8-flash,212.716,222.337,1511,157.1
3,Code Generation,deepseek-v4.1-flash-maas,0.877,13.526,1152,91.1
4,Summarization,gemini-3.8-flash,237.102,238.953,189,102.1
5,Summarization,deepseek-v4.1-flash-maas,2.343,12.818,126,12.0


### 📈 Benchmark-Auswertung: Gemini 3.8 Flash vs. DeepSeek V4.1 Flash MaaS

| Kategorie | Modell | TTFT (s) | Gesamtzeit (s) | Output Tokens | Durchsatz (TPS) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Reasoning & Math** | `gemini-3.8-flash`<br>`deepseek-v4.1-flash-maas` | 145.0 s<br>**1.45 s** | 153.0 s<br>**15.2 s** | **1.036**<br>712 | **129.6 TPS**<br>51.7 TPS |
| **Code Generation** | `gemini-3.8-flash`<br>`deepseek-v4.1-flash-maas` | 212.7 s<br>**0.88 s** | 222.3 s<br>**13.5 s** | **1.511**<br>1.152 | **157.1 TPS**<br>91.1 TPS |
| **Summarization** | `gemini-3.8-flash`<br>`deepseek-v4.1-flash-maas` | 237.1 s<br>**2.34 s** | 239.0 s<br>**12.8 s** | **189**<br>126 | **102.1 TPS**<br>12.0 TPS |

---

#### 🔍 Kern-Erkenntnisse & Bewertung

1. **TTFT (Time-to-First-Token) & Interaktivität:**
   * **DeepSeek V4.1 Flash** glänzt bei der Reaktionszeit: Mit **sub-2s TTFT** (0.88s bei Code!) beginnt das Streaming praktisch sofort. Für synchrone Chatbots und kundennahe UIs unschlagbar reaktiv.
   * **Gemini 3.8 Flash** zeigt eine sehr lange Initial-Latenz (145s – 237s). Ursachen sind das native **Deep Think / Reasoning**, bei dem das Modell vorab interne Denk-Tokens evaluiert, kombiniert mit Queueing auf dem globalen Preview-Endpunkt.

2. **Generierungs-Durchsatz (Tokens/s):**
   * Sobald Gemini 3.8 Flash zu schreiben beginnt, ist die reine Inferenz extrem schnell: **130 – 157 TPS** (fast doppelt so schnell wie DeepSeeks 51 – 91 TPS).

3. **Detailgrad & Token-Volumen:**
   * Gemini 3.8 Flash liefert 30–50 % mehr Tokens und formuliert deutlich ausführlichere Lösungen, Code-Kommentare und Erklärungen.

**Fazit & Empfehlung für die Praxis:**
* **Client-Facing Chatbots:** **DeepSeek V4.1 Flash** (sofortiges Nutzer-Feedback, Gesamtzeit ~13–15s).
* **Asynchrone Agenten-Pipelines / Complex Reasoning (`agents.ipynb`):** **Gemini 3.8 Flash** (überlegener Token-Durchsatz, gründliche Analyse, tiefere logische Verknüpfung).


In [ ]:
# @title Vertex AI - Deployment Recommendation
#################################################

from vertexai import model_garden

# Model Garden Identifier (Format: google/gemma4@gemma-4-26b-a4b-it oder google/gemma4@gemma-4-31b-it)
MODEL_ID = "google/gemma4@gemma-4-26b-a4b-it"

# Model Garden Objekt initialisieren
model = model_garden.OpenModel(MODEL_ID)

# Empfohlene/verifizierte Deployment-Optionen abrufen (concise=True für lesbare Übersicht)
deploy_options = model.list_deploy_options(concise=True)
print(deploy_options)

[Option 1: vLLM 256K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
    machine_type="g4-standard-48",
    accelerator_type="NVIDIA_RTX_PRO_6000",
    accelerator_count=1,

[Option 2: vLLM 256K context spec decode]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20260505_0916_RC01",
    machine_type="g4-standard-48",
    accelerator_type="NVIDIA_RTX_PRO_6000",
    accelerator_count=1,

[Option 3: vLLM 256K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
    machine_type="g4-standard-96",
    accelerator_type="NVIDIA_RTX_PRO_6000",
    accelerator_count=2,

[Option 4: vLLM 256K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
    machine_type="a3-highgpu-1g",
    acceler

In [ ]:
# @title Vertex AI - Deployment Script ⚠️ BE CAREFUL (Deployment incurs costs!)

"""
Recommendation: NVIDIA RTX Pro 6000 (Blackwell) with VRAM	96 GB
We prefer NVIDIA L4 (Ada Lovelace) with VRAM 24 GB. Capacity: The L4 has 24 GB of GDDR6 memory.
The Problem: The Gemma 4 26B model in full precision (BF16) requires roughly 52 GB of VRAM just to load the weights. It will not fit on a single L4 in that state.
The Solution (Quantization): To run this on an L4, you must use 4-bit quantization (like AWQ or INT4). In 4-bit mode, the weights take up only ~15–18 GB, which fits comfortably on a single L4.
The Catch: You will likely need to reduce your context window significantly (e.g., from 256K down to 8K or 16K) to stay within the L4's 24GB limit.
"""


"""
import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-26b-a4b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-26b-a4b-it-mg-one-click-deploy",
  model_display_name="gemma-4-26b-a4b-it-1777568163447",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)
"""

import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-31b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-31b-it-mg-one-click-deploy",
  model_display_name="gemma-4-31b-it-1777570278133",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)